<div dir="rtl">

# MCMC בקוסמולוגיה: מקו ישר ועד נתוני Pantheon+

**מה נלמד במחברת הזו.** נבנה צעד אחר צעד את כל המכונה של הסקה בייסיאנית עם דוגם **`emcee`**, קודם על הדוגמה הפשוטה ביותר שקיימת, **קו ישר**, שבה אפשר לפתור הכול אנליטית ולבדוק שהדגימה נכונה. ואז נחליף את הקו הישר בנתונים אמיתיים: **1590 סופרנובות מקטלוג Pantheon+**, עם מטריצת שונות מלאה. *המודל ישתנה, המכונה תישאר.*

**מבנה:**

| חלק | נושא      | מה לומדים                                                                                     |
| --- | --------- | --------------------------------------------------------------------------------------------- |
| א   | קו ישר    | נראות, מוקדמת, `emcee`, אבחון (τ, R̂), corner plot, מרכוז, מרג'ינליזציה, AIC/BIC              |
| ב   | Pantheon+ | מטריצת שונות, מרג'ינליזציה אנליטית על ההיסט, $\Lambda$CDM, $w$CDM, יקום עקום, פרמטרים נגזרים |

**מה צריך.** `numpy`, `scipy`, `matplotlib`, `pandas`, `emcee`, `corner`. התקנה: `pip install numpy scipy matplotlib pandas emcee corner`. זמן ריצה כולל: כ-10 דקות. נתוני Pantheon+ (כ-34MB) יורדים אוטומטית מ-GitHub בהרצה הראשונה, ונשמרים בתיקייה המקומית.

</div>

<div dir="rtl">

## 0. הכנות

נייבא את הספריות. שימו לב לשלושה דברים: (1) `np.random.default_rng(seed)` נותן מחולל מספרים אקראיים עם **זרע קבוע**, כדי שאפשר יהיה לשחזר את התוצאות ולחפש באגים; (2) `emcee` הוא הדוגם עצמו; (3) `corner` מצייר את ה-corner plot.

> **כלל זהב:** בכל סימולציה אקראית, כולל MCMC, שומרים את הזרע ואת השרשראות. זה חוסך ימים של עבודה.

</div>

In [ ]:
import os, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import emcee, corner
from scipy.optimize import minimize
from scipy.integrate import cumulative_trapezoid, trapezoid

plt.rcParams.update({"font.size":12, "axes.spines.top":False, "axes.spines.right":False, "figure.dpi":110})
rng = np.random.default_rng(42)            # זרע קבוע
print("emcee", emcee.__version__, "| corner", corner.__version__)

<div dir="rtl">

# חלק א: קו ישר

## 1. הבעיה

יש לנו $N$ נקודות $(x_i, y_i)$ עם שגיאה גאוסית ידועה $\sigma$, ומודל ליניארי בשני פרמטרים:

$$y = a + b\,x, \qquad \boldsymbol\theta=(a,b).$$

זה בדיוק החישוב של האבל: $b$ הוא השיפוע שמעניין אותנו ו-$a$ הוא היסט שלא. בקוסמולוגיה של סופרנובות $x=z$ ו-$y=m_B-5\log_{10}z$, ולכן $a$ קשור לבהירות המוחלטת ול-$H_0$, ו-$b\simeq1.086\,(1-q_0)$.

**למה נתונים סינתטיים?** כי אנחנו *יודעים* את האמת, $(a,b)=(23.86,\ 1.683)$, ולכן אפשר לבדוק אם הדוגם מחזיר אותה. כך בודקים כל קוד MCMC לפני שמפעילים אותו על נתונים אמיתיים.

</div>

In [ ]:
N, sigma = 40, 0.15
a_true, b_true = 23.86, 1.683
x = np.sort(rng.uniform(0.01, 0.15, N))                 # ההסחות של הסופרנובות
y = a_true + b_true*x + rng.normal(0, sigma, N)         # מודל + רעש גאוסי

fig, ax = plt.subplots(figsize=(6.4,4))
ax.errorbar(x, y, sigma, fmt="o", color="k", ms=4, lw=1)
xx = np.linspace(0, 0.16, 10); ax.plot(xx, a_true+b_true*xx, color="C3", label="truth")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(); plt.show()

<div dir="rtl">

## 2. הפתרון האנליטי: מה אנחנו אמורים לקבל

עם שגיאות גאוסיות בלתי-תלויות ומוקדמת אחידה, $-2\ln\mathcal L=\chi^2$, ו-$\chi^2$ **ריבועי** ב-$(a,b)$. לכן המאוחרת גאוסית *בדיוק*, ואפשר לפתור בעיפרון. נגדיר את מטריצת התכנון $A$ (עמודה של אחדות ועמודה של $x$) ואת $F=A^{\top}WA$ עם $W=\sigma^{-2}\mathbb 1$:

$$F\,\hat{\boldsymbol\theta}=A^{\top}Wy,\qquad \mathsf C=F^{-1},\qquad \rho_{ab}=\frac{C_{ab}}{\sqrt{C_{aa}C_{bb}}}.$$

$\mathsf C$ היא מטריצת השונות המשותפת של המאוחרת. האלכסון נותן את השגיאות *הממורג'נות*, וה-$\rho$ את הקורלציה. **המטרה** של ה-MCMC: לדגום ממאוחרת גאוסית עם הממוצע והשונות האלה.

</div>

In [ ]:
A = np.vstack([np.ones(N), x]).T
F = A.T @ A / sigma**2
theta_hat = np.linalg.solve(F, A.T @ y / sigma**2)
C = np.linalg.inv(F)
sa, sb = np.sqrt(np.diag(C)); rho = C[0,1]/(sa*sb)
chi2_min = np.sum(((y - A@theta_hat)/sigma)**2)
print(f"a = {theta_hat[0]:.3f} ± {sa:.3f}")
print(f"b = {theta_hat[1]:.3f} ± {sb:.3f}")
print(f"rho_ab = {rho:.3f}   chi2_min = {chi2_min:.1f}  (dof = {N-2})")

<div dir="rtl">

שימו לב ל-$\rho_{ab}\approx-0.9$: כל ה-$x$ חיוביים ומרוכזים הרחק מהראשית, ולכן הקו ״מסתובב״ סביב מרכז הנתונים. להגדיל את השיפוע פירושו להנמיך את החותך. **זו הקורלציה שתאט את הדוגם**, ונחזור אליה בסעיף 9.

</div>

<div dir="rtl">

## 3. הפונקציה `log_probability`: הלב של כל MCMC

`emcee` צריך פונקציה אחת: **לוג של המאוחרת הלא-מנורמלת** כפונקציה של וקטור הפרמטרים $\boldsymbol\theta$. מה שחשוב לדעת:

1.  **למה לוגריתם?** מכפלות של הרבה גורמים קטנים עלולות לגלוש מתחת לדיוק המחשב. בלוג הן הופכות לסכומים.
2.  **הנרמול מתבטל.** הדוגם משתמש רק ביחס $P(\boldsymbol\theta_{\rm new})/P(\boldsymbol\theta)$, ולכן הקבועים, כולל העדות $P(D)$, נשמטים. זה הקסם של MCMC: אפשר לדגום ממאוחרת שאיננו יודעים לנרמל.
3.  **מוקדמת אחידה:** $\ln P=0$ בתוך התחום, ו-$-\infty$ מחוצה לו. החזרת $-\infty$ פירושה ״אסור״: הצעד נדחה תמיד.
4.  **הנראות:** $\ln\mathcal L=-\tfrac12\chi^2$.

הפונקציה מקבלת גם את הנתונים כארגומנטים נוספים (`args`), כך שאין משתנים גלובליים.

</div>

In [ ]:
def log_prior(theta):
    a, b = theta
    if 20 < a < 28 and -10 < b < 10:          # מוקדמת אחידה בתחום רחב
        return 0.0
    return -np.inf

def log_likelihood(theta, x, y, sigma):
    a, b = theta
    resid = (y - a - b*x) / sigma
    return -0.5 * np.sum(resid**2)            # נשמיט קבועים

def log_probability(theta, x, y, sigma):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf                         # מחוץ לתחום: אסור
    return lp + log_likelihood(theta, x, y, sigma)

print("ב-theta_hat:     ", log_probability(theta_hat, x, y, sigma))
print("רחוק מהאמת:      ", log_probability([23.0, 5.0], x, y, sigma))
print("מחוץ למוקדמת:    ", log_probability([30.0, 0.0], x, y, sigma))

<div dir="rtl">

## 4. איתחול ההולכים (walkers)

`emcee` הוא **דוגם אנסמבל** (Goodman & Weare): במקום שרשרת אחת מריצים `nwalkers` ״הולכים״ במקביל, וכל אחד מציע צעד בעזרת הולך אחר (*stretch move*). היתרון הגדול: הדוגם **אינווריאנטי תחת טרנספורמציות אפיניות**, ולכן מתמודד היטב עם ניוונים בצורת סיגר, כמו הקורלציה $\rho\approx-0.9$ שלנו, בלי שנצטרך לכוונן גודל צעד.

כללי אצבע: `nwalkers` לפחות פעמיים מספר הפרמטרים (בדרך כלל 16–64), ומתחילים ב״כדור קטן״ סביב נקודה טובה, כלומר סביב המקסימום של הנראות.

</div>

In [ ]:
nll = lambda th: -log_probability(th, x, y, sigma)
best = minimize(nll, [23.5, 1.0], method="Nelder-Mead").x       # המקסימום של הנראות
print("best fit:", best)

ndim, nwalkers = 2, 32
p0 = best + 1e-2*np.array([sa, sb])*rng.normal(size=(nwalkers, ndim))   # כדור קטן סביב המקסימום
print("צורת p0:", p0.shape)

<div dir="rtl">

## 5. ההרצה

ניצור את ה-`EnsembleSampler` ונריץ. כל צעד מזיז את *כל* ההולכים, ולכן אחרי `nsteps` צעדים נקבל שרשרת בצורה `(nsteps, nwalkers, ndim)`.

`sampler.acceptance_fraction` הוא שיעור הקבלה של כל הולך. ל-`emcee` ערכים טיפוסיים הם **0.2–0.7** (בממדים נמוכים, כמו כאן, קרוב לקצה העליון). אם הוא קרוב ל-0 או ל-1, משהו לא בסדר בפונקציה או בתנאי ההתחלה. *(קבלה גבוהה אינה הצלחה: היא יכולה להעיד על צעדים זעירים והליכה איטית. מה שחשוב הוא τ שנחשב בהמשך.)*

</div>

In [ ]:
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=(x, y, sigma))
t0 = time.time()
sampler.run_mcmc(p0, 6000, progress=False)
print(f"זמן ריצה: {time.time()-t0:.1f} s")
chain = sampler.get_chain()                       # (nsteps, nwalkers, ndim)
print("צורת השרשרת:", chain.shape)
print("שיעור קבלה ממוצע:", np.mean(sampler.acceptance_fraction).round(3))

<div dir="rtl">

## 6. אבחון: האם אפשר להאמין לשרשרת?

שלושה כלים, ושלושתם חיוניים *לפני* שמדווחים תוצאה:

1.  **גרף עקבות (trace):** כל הולך כקו. אחרי תקופה קצרה כולם צריכים להתערבב סביב אותו אזור, בלי מגמות. החלק הראשון, ה-**burn-in**, תלוי בנקודת ההתחלה, ומשליכים אותו.
2.  **זמן אוטוקורלציה $\tau$:** דגימות עוקבות תלויות זו בזו. מספר הדגימות *הבלתי-תלויות* הוא $N_{\rm eff}\approx N_{\rm steps}\,N_{\rm walkers}/\tau$, וזה המספר שקובע את הדיוק, לא $N$. `emcee` ממליץ שהשרשרת תהיה ארוכה מ-$\sim50\tau$.
3.  **Gelman–Rubin $\hat R$:** משווה שונות *בין* ריצות בלתי-תלויות לשונות *בתוך* ריצה. דורשים $\hat R-1\<0.01$.

</div>

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8,5), sharex=True)
for i, lab in enumerate(["a", "b"]):
    axes[i].plot(chain[:, :, i], color="k", alpha=0.25, lw=0.6)
    axes[i].set_ylabel(lab)
axes[1].set_xlabel("step"); plt.show()

tau = sampler.get_autocorr_time(tol=0)
print("τ =", tau.round(1), "  → N_steps/τ =", (6000/tau).round(0))

<div dir="rtl">

העקבות נראות כמו ״קרפיט שערות״ ללא מגמות: הדוגם התערבב. עכשיו נשליך burn-in של כ-$5\tau$, ו-נשטח (flatten) את הולכים והצעדים לרשימה אחת של דגימות. *Thinning*, שמירה של דגימה אחת מכל כמה, אינו משפר את הדיוק, הוא רק חוסך זיכרון, ולכן נעשה אותו בעדינות, בערך $\tau/2$.

</div>

In [ ]:
burn = int(5*tau.max()); thin = max(1, int(tau.max()/2))
flat = sampler.get_chain(discard=burn, thin=thin, flat=True)
n_eff = (6000-burn)*nwalkers/tau.max()
print(f"burn-in = {burn}, thin = {thin}, דגימות = {flat.shape[0]}, N_eff ≈ {n_eff:.0f}")

<div dir="rtl">

**$\hat R$ מ-4 ריצות בלתי-תלויות.** נריץ ארבע פעמים, מנקודות התחלה **מפוזרות** ($\pm6\sigma$ סביב המקסימום), ונחשב את

$$\hat R=\sqrt{\frac{\frac{n-1}{n}W+\left(1+\frac1M\right)B}{W}},\qquad B=\text{שונות בין ממוצעי ה-}M\text{ ריצות},\ \ W=\text{שונות ממוצעת בתוך ריצה}.$$

> **הערה:** ההולכים של `emcee` אינם בלתי-תלויים זה בזה, ולכן כאן מתייחסים לכל *ריצה* כשרשרת אחת. עבור ניתוח כמותי עדין משתמשים ב-`arviz`.

</div>

In [ ]:
def gelman_rubin(runs):
    # runs: רשימה של מערכים (דגימות, ndim) מריצות בלתי-תלויות
    M = len(runs); n = min(len(r) for r in runs); runs = [r[:n] for r in runs]
    means = np.array([r.mean(0) for r in runs])
    W = np.mean([r.var(0, ddof=1) for r in runs], axis=0)       # שונות בתוך ריצה
    B = means.var(0, ddof=1)                                       # שונות בין ממוצעי הריצות
    return np.sqrt(((n-1)/n*W + (1+1/M)*B) / W)

runs = []
for seed in range(4):
    r = np.random.default_rng(100+seed)
    p0r = best + 6*np.array([sa, sb])*r.uniform(-1, 1, size=(16, 2))        # התחלה מפוזרת
    s = emcee.EnsembleSampler(16, 2, log_probability, args=(x, y, sigma))
    s.run_mcmc(p0r, 4000, progress=False)
    runs.append(s.get_chain(discard=1000, flat=True))
print("R̂ (a, b) =", gelman_rubin(runs).round(4))

<div dir="rtl">

## 7. ה-corner plot וקריאתו

זו ההצגה הסטנדרטית של שרשרת: משולש של $N\times N$ פאנלים.

  - **באלכסון:** היסטוגרמה של פרמטר בודד, כלומר ההתפלגות **השולית** (מרג'ינליזציה = התעלמות מכל שאר העמודות).
  - **מחוץ לאלכסון:** ההטלה הדו-ממדית של זוג פרמטרים. אליפסה משופעת היא קורלציה, וככל שהיא דקה יותר זה ניוון חזק יותר.
  - **הקווים האנכיים המקווקווים:** חציון ואחוזונים 16 ו-84, כלומר ה-$\pm$ המקובל.
  - **קווי המתאר:** כאן אנחנו מבקשים במפורש שיכילו $68\%$ ו-$95\%$ מהמסה. *זו לא תמיד ברירת המחדל\!* בחבילות רבות הברירה היא $11.8\%, 39.3\%, 67.5\%, 86.5\%$ (רדיוסים $0.5,1,1.5,2\sigma$ של גאוסיאן דו-ממדי), ולכן ״$1\sigma$״ שלהן סוגר רק $39\%$. **תמיד בדקו מה אחוז המסה שהקווים סוגרים.**

הקווים האדומים הם הפתרון האנליטי מסעיף 2.

</div>

In [ ]:
fig = corner.corner(flat, labels=["a", "b"], truths=list(theta_hat), truth_color="C3",
                    levels=(0.68, 0.95), quantiles=[0.16, 0.5, 0.84], show_titles=True,
                    title_fmt=".3f", color="C0", plot_datapoints=False, fill_contours=True, smooth=1.0)
plt.show()

q = np.percentile(flat, [16, 50, 84], axis=0)
print("MCMC:      a = %.3f +%.3f -%.3f   b = %.3f +%.3f -%.3f" % (q[1,0], q[2,0]-q[1,0], q[1,0]-q[0,0], q[1,1], q[2,1]-q[1,1], q[1,1]-q[0,1]))
print("אנליטי:    a = %.3f ± %.3f        b = %.3f ± %.3f" % (theta_hat[0], sa, theta_hat[1], sb))
print("קורלציה:   MCMC %.3f   אנליטי %.3f" % (np.corrcoef(flat.T)[0,1], rho))

<div dir="rtl">

ההתאמה בין הדגימה לפתרון האנליטי, בשגיאות, בקורלציה ובצורה, היא **הבדיקה** שהדוגם שלנו נכון. *כך אפשר לסמוך על השיטה כשאין עם מה להשוות.*

</div>

<div dir="rtl">

## 8. אי-ודאות אמיתית: קווים מתוך המאוחרת

כל נקודה בשרשרת היא קו אפשרי. נצייר 60 קווים אקראיים. שימו לב שכולם עוברים דרך ״צוואר״ צר סביב $\bar x$ ונפתחים בקצוות, וזה בדיוק מקור הקורלציה השלילית.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(6.4,4))
ax.errorbar(x, y, sigma, fmt="o", color="k", ms=4, lw=1, zorder=3)
for a_, b_ in flat[rng.integers(len(flat), size=60)]:
    ax.plot(xx, a_ + b_*xx, color="C0", alpha=0.15)
ax.axvline(x.mean(), color="gray", ls=":"); ax.text(x.mean()+0.002, y.min(), r"$\bar x$", color="gray")
ax.set_xlabel("x"); ax.set_ylabel("y"); plt.show()

<div dir="rtl">

## 9. פרמטריזציה חכמה: מרכוז

הקורלציה $\rho=-\bar x/\sqrt{\langle x^2\rangle}$ באה מכך שהנתונים רחוקים מהראשית ביחס לפיזורם. אם נמדוד את $x$ מהממוצע, $x'=x-\bar x$, הקורלציה מתאפסת, והחותך החדש $a'=a+b\bar x$ הוא ערך הקו ב-$\bar x$ ונמדד הכי טוב ($\sigma_{a'}=\sigma/\sqrt N$).

**אותה מאוחרת בדיוק, פרמטריזציה אחרת.** האם זה משנה לדוגם? בדוגם אנסמבל כמעט לא (הוא אפיני-אינווריאנטי), אבל בדוגמים אחרים, למשל Metropolis עם צעדים אלכסוניים, זה משנה את $\tau$ בפקטור של 3 ויותר. זו הסיבה שבקודים קוסמולוגיים בוחרים פרמטר בסיס כמו $\theta_s$ במקום $H_0$.

</div>

In [ ]:
xbar = x.mean()
def log_prob_centered(theta, x, y, sigma):
    ap, b = theta                      # a' = a + b*xbar
    return log_probability([ap - b*xbar, b], x, y, sigma)

p0c = np.array([best[0] + best[1]*xbar, best[1]]) + 1e-2*np.array([sigma/np.sqrt(N), sb])*rng.normal(size=(nwalkers, 2))
sc = emcee.EnsembleSampler(nwalkers, 2, log_prob_centered, args=(x, y, sigma))
sc.run_mcmc(p0c, 6000, progress=False)
flat_c = sc.get_chain(discard=burn, flat=True)
print("τ (a', b)  =", sc.get_autocorr_time(tol=0).round(1), "   לעומת (a, b) =", tau.round(1))
print("קורלציה (a', b) =", np.corrcoef(flat_c.T)[0,1].round(3), "   σ_a' =", flat_c[:,0].std().round(4), " (תיאוריה:", round(sigma/np.sqrt(N),4), ")")

<div dir="rtl">

## 10. למה מרג'ינליזציה ולא קיבוע

נניח שנקבע את $a$ בערך הטוב ביותר ונדגום רק את $b$. השגיאה שנקבל קטנה מדי בפקטור $1/\sqrt{1-\rho^2}$. פרמטר מטרד שמתואם עם הפרמטר שמעניין **חייב להיות ממורג'ן, לא מקובע**.

</div>

In [ ]:
sb_cond = 1/np.sqrt(F[1,1])                       # שגיאה על b כש-a מקובע
print(f"σ_b ממורג'ן (אנליטי) = {sb:.3f}")
print(f"σ_b עם a מקובע       = {sb_cond:.3f}   → יחס {sb/sb_cond:.2f}  (תיאוריה: 1/√(1-ρ²) = {1/np.sqrt(1-rho**2):.2f})")
s1 = emcee.EnsembleSampler(8, 1, lambda t: log_probability([theta_hat[0], t[0]], x, y, sigma))
s1.run_mcmc(theta_hat[1] + 1e-2*sb*rng.normal(size=(8,1)), 4000, progress=False)
print(f"σ_b מדגימה עם a מקובע = {s1.get_chain(discard=500, flat=True).std():.3f}")

<div dir="rtl">

## 11. האם הקו מספיק? השוואת מודלים

נתאים לאותם נתונים גם פרבולה, $y=a+bx+cx^2$. פרמטר נוסף תמיד משפר את $\chi^2$, ולכן צריך ״קנס״ על מורכבות:

$$\mathrm{AIC}=\chi^2_{\min}+2k,\qquad \mathrm{BIC}=\chi^2_{\min}+k\ln N .$$

שניהם נובעים מאותו עיקרון (התער של אוקאם), אבל AIC עונה על ״איזה מודל ינבא טוב יותר נתונים חדשים״ ו-BIC על ״איזה מודל סביר יותר שיצר את הנתונים״. בקירוב $\ln B_{12}\simeq-\Delta\mathrm{BIC}/2$. **סולם ג'פריס הוא מוסכמה**, והעדות המדויקת תלויה בטווחי המוקדמת.

</div>

In [ ]:
A3 = np.vstack([np.ones(N), x, x**2]).T
t3 = np.linalg.lstsq(A3, y, rcond=None)[0]; chi3 = np.sum(((y - A3@t3)/sigma)**2)
d_chi2 = chi2_min - chi3
print(f"Δχ² = {d_chi2:.2f}   ΔAIC = {(chi3+6)-(chi2_min+4):+.2f}   ΔBIC = {(chi3+3*np.log(N))-(chi2_min+2*np.log(N)):+.2f}   (ln N = {np.log(N):.2f})")
print("ערך חיובי = לרעת הפרבולה. הנתונים הוגרלו מקו, ולכן זו התשובה הנכונה.")

<div dir="rtl">

### סיכום חלק א, ותרגילים

**מה בנינו:** נתונים ← מודל ← `log_probability` ← הולכים ← `run_mcmc` ← אבחון ← corner. **כל** ניתוח קוסמולוגי בנוי כך, גם כשהמודל מורכב והנראות איטית.

**תרגילים לבדיקה עצמית:**

1.  הכפילו את טווח ה-$x$ (בלי להגדיל את $N$) וצפו מה קורה ל-$\sigma_b$. מה זה אומר על תכנון סקר סופרנובות?
2.  החליפו את המוקדמת האחידה על $a$ במוקדמת צרה ($\pm0.1$) סביב ערך שגוי. מה קורה למאוחרת?
3.  הקטינו את `nwalkers` ל-4 והגדילו את `nsteps` ל-60000. האם התוצאה זהה? ומה קורה ל-$\tau$?
4.  הוסיפו שגיאה גדולה פי 3 לנקודה אחת (outlier) וראו איך זה משפיע על ה-corner.

</div>

<div dir="rtl">

# חלק ב: נתוני Pantheon+

עכשיו נחליף את הקו הישר בנתונים אמיתיים. **שלושה דברים משתנים, והמכונה נשארת:**

1.  **המודל לא ליניארי** ב-$\Omega_m,w$ (הם נכנסים דרך אינטגרל), ולכן המאוחרת אינה גאוסית וה-MCMC כבר אינו מותרות.
2.  **ההיסט $\mathcal M$** (שמכיל את הבהירות המוחלטת ואת $H_0$) *כן* ליניארי, ולכן אפשר למרג'ן עליו **אנליטית**.
3.  **השגיאות מתואמות:** $\chi^2=\mathbf r^{\top}\mathsf C^{-1}\mathbf r$ ולא סכום של $\sigma_i^{-2}$ בלתי-תלויים.

</div>

<div dir="rtl">

## 12. הנתונים

Pantheon+ מכיל 1701 עקומות אור של סופרנובות מסוג Ia (1550 סופרנובות שונות, חלקן נמדדו בכמה סקרים). נשתמש בשלוש עמודות:

  - `zHD`: ההסחה ל-Hubble diagram, **מתוקנת לשדה המהירויות** ולתנועה שלנו. משמשת ב-$D_M$.
  - `zHEL`: ההסחה ההליוצנטרית. משמשת בגורם $(1+z)$ של $d_L$ (שקובע את אובדן האנרגיה בפועל).
  - `m_b_corr`: המגניטודה הנראית **המתוקנת** (תיקוני מתיחה וצבע של טריפ כבר הוכנסו).

ונחתוך $z_{\rm HD}\>0.01$ כדי להימנע מהסולמות שבהם מהירויות פקוליאריות דומיננטיות. מטריצת השונות (סטטיסטית + שיטתית) מורדת מאותו מאגר.

</div>

In [ ]:
BASE = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
def fetch(remote, local):
    if not os.path.exists(local):
        print("מוריד", local, "...")
        urllib.request.urlretrieve(BASE + remote, local)
    return local

f_dat = fetch("Pantheon%2BSH0ES.dat", "Pantheon+SH0ES.dat")
f_cov = fetch("Pantheon%2BSH0ES_STAT%2BSYS.cov", "Pantheon+SH0ES_STAT+SYS.cov")

df = pd.read_csv(f_dat, sep=r"\s+")
if os.path.exists("pp_cov.npy"):
    cov_all = np.load("pp_cov.npy")
else:
    cov_all = np.loadtxt(f_cov, skiprows=1).reshape(len(df), len(df)); np.save("pp_cov.npy", cov_all)

mask = (df.zHD > 0.01).values
d = df[mask]
zhd, zhel, mb = d.zHD.values, d.zHEL.values, d.m_b_corr.values
Csn = cov_all[np.ix_(mask, mask)]
print(f"בקטלוג: {len(df)} עקומות אור | אחרי החיתוך z_HD>0.01: {mask.sum()}")
print(f"טווח הסחה: {zhd.min():.4f} – {zhd.max():.3f}")
d[["CID","zHD","zHEL","m_b_corr","m_b_corr_err_DIAG"]].head()

<div dir="rtl">

## 13. מטריצת השונות: למה לא מספיק $\sigma_i$

האלכסון של $\mathsf C$ הוא השגיאה של כל סופרנובה, אבל **האיברים מחוץ לאלכסון** הם שמבטאים את השגיאות השיטתיות: אותו כיול פוטומטרי משפיע על סופרנובות רבות יחד. התעלמות מהם נותנת שגיאות קטנות מדי ו-$\chi^2$ שגוי. ננוודא שהמטריצה סימטרית וחיובית-מוגדרת (תנאי לכך שהיא חוקית), ונחשב את ההופכית פעם אחת.

> **ב-MCMC אסור להפוך מטריצה בכל צעד.** מחשבים $\mathsf C^{-1}$ פעם אחת לפני ההרצה, ובכל צעד רק מכפילים וקטור. זה ההבדל בין שניות לשעות.

</div>

In [ ]:
Csn = 0.5*(Csn + Csn.T)                                 # סימטריזציה (שגיאות עיגול בקובץ)
np.linalg.cholesky(Csn)                                  # נכשל אם המטריצה אינה חיובית-מוגדרת
Cinv = np.linalg.inv(Csn)
ones = np.ones(len(mb)); s_vec = Cinv @ ones; S_sum = ones @ s_vec

diag_err = np.sqrt(np.diag(Csn))
off = np.abs(Csn[~np.eye(len(mb), dtype=bool)])
print(f"שגיאה טיפוסית (אלכסון): {np.median(diag_err):.3f} mag | איבר טיפוסי מחוץ לאלכסון: {np.median(off):.1e} mag² ")
fig, ax = plt.subplots(figsize=(5.2,4.4))
im = ax.imshow(np.log10(np.abs(Csn)+1e-9), cmap="viridis"); plt.colorbar(im, label=r"$\log_{10}|C_{ij}|$"); ax.set_title("covariance (STAT+SYS)"); plt.show()

<div dir="rtl">

## 14. המודל הפיזיקלי

ניתן לכתוב את המרחק הזוהר בצורה שבה כל התלות ב-$H_0$ יושבת בפרה-פקטור:

$$d_L=(1+z_{\rm hel})\,D_M(z_{\rm HD}),\qquad D_M=\frac{c}{H_0}\int_0^{z}\frac{dz'}{E(z')},\qquad \mu=5\log_{10}\frac{d_L}{\mathrm{Mpc}}+25,$$

$$E^2(z)=\Omega_m(1+z)^3+(1-\Omega_m)(1+z)^{3(1+w)}\quad(\text{שטוח}),\qquad E^2=\Omega_m(1+z)^3+\Omega_k(1+z)^2+\Omega_\Lambda\quad(\text{עקום}).$$

$H_0=70$ מקובע, כי *סופרנובות לבדן אינן מודדות את $H_0$* (ראו סעיף 15). **לביצועים:** מחשבים את האינטגרל פעם אחת על רשת צפופה של $z$ בעזרת סכום מצטבר (`cumulative_trapezoid`), ואז מפרשים (`np.interp`) לכל 1590 הסופרנובות. זה מהיר פי מאות מ-`quad` לכל סופרנובה.

</div>

In [ ]:
c_light, H0_fix = 299792.458, 70.0                    # km/s, km/s/Mpc
zg = np.linspace(0, 2.4, 4801)                         # רשת z צפופה

def DM_grid(Om, w=-1.0, OL=None):
    # מרחק נע רוחבי D_M(z) על הרשת. OL=None → יקום שטוח עם w; אחרת יקום עקום עם Λ.
    if OL is None:
        E2 = Om*(1+zg)**3 + (1-Om)*(1+zg)**(3*(1+w))
        Ok = 0.0
    else:
        Ok = 1 - Om - OL
        E2 = Om*(1+zg)**3 + Ok*(1+zg)**2 + OL
    if E2.min() <= 0: return None                      # פתרון לא פיזיקלי
    f = 1/np.sqrt(E2)
    DC = (c_light/H0_fix) * cumulative_trapezoid(f, zg, initial=0)
    if abs(Ok) < 1e-10: return DC
    k = np.sqrt(abs(Ok)); DH = c_light/H0_fix
    return DH/k*np.sinh(k*DC/DH) if Ok > 0 else DH/k*np.sin(k*DC/DH)

def mu_model(Om, w=-1.0, OL=None):
    DM = DM_grid(Om, w, OL)
    if DM is None: return None
    dl = (1+zhel)*np.interp(zhd, zg, DM)
    return 5*np.log10(dl) + 25
print("μ(z=0.5), Ωm=0.3, ΛCDM:", np.round(5*np.log10((1+0.5)*np.interp(0.5, zg, DM_grid(0.3)))+25, 3))

<div dir="rtl">

### דיאגרמת האבל

נצייר את הנתונים מול כמה מודלים: Einstein–de Sitter $(\Omega_m=1)$, יקום פתוח בלי $\Lambda$, ו-$\Lambda$CDM. כדי שההשוואה תהיה הוגנת, לכל מודל מוסיפים את ההיסט הטוב ביותר $\hat{\mathcal M}$ (שנגזר בסעיף הבא). *הסיפור כולו נמצא בשארית:* EdS סוטה עד כ-$0.4$ mag, ו-$\Lambda$CDM מתאים.

</div>

In [ ]:
def best_offset(mu):
    r0 = mb - mu
    return (s_vec @ r0)/S_sum                          # M^ = B/C

def binned(zz, res, nb=14):
    edges = np.quantile(zz, np.linspace(0,1,nb+1)); idx = np.digitize(zz, edges[1:-1])
    return np.array([zz[idx==i].mean() for i in range(nb)]), np.array([res[idx==i].mean() for i in range(nb)])

models = {"EdS (Ωm=1)": mu_model(1.0, OL=0.0), "open (0.3, 0)": mu_model(0.3, OL=0.0), "ΛCDM (0.33)": mu_model(0.33)}
ref = models["ΛCDM (0.33)"]; ref = ref + best_offset(ref)
fig, ax = plt.subplots(2, 1, figsize=(7,6.2), sharex=True, gridspec_kw=dict(height_ratios=[2,1]))
ax[0].plot(zhd, mb, ".", ms=2.5, color="gray", alpha=0.6); ax[0].set_xscale("log"); ax[0].set_ylabel(r"$m_B^{\rm corr}$")
zb, rb = binned(zhd, mb - ref); ax[1].plot(zb, rb, "ko", ms=4)
for name, mu in models.items():
    mm = mu + best_offset(mu); o = np.argsort(zhd)
    ax[0].plot(zhd[o], mm[o], lw=1.8, label=name)
    zb2, rb2 = binned(zhd, mm - ref); ax[1].plot(zb2, rb2, lw=1.8)
ax[1].axhline(0, color="k", lw=0.8); ax[1].set_ylabel("binned data − ΛCDM  [mag]"); ax[1].set_xlabel(r"$z_{\rm HD}$"); ax[0].legend(); plt.show()

<div dir="rtl">

## 15. מרג'ינליזציה אנליטית על ההיסט $\mathcal M$

המגניטודה הנראית היא $m_B=\mathcal M+\mu(z;\boldsymbol\theta)$, ו-$\mathcal M=M_B+5\log_{10}(c/H_0\,\mathrm{Mpc})+25$ מכיל את הבהירות המוחלטת **ואת** $H_0$ בצירוף אחד. **לכן סופרנובות לבדן אינן מודדות את $H_0$**, ושינוי של $H_0$ שקול בדיוק להזזת $M_B$. נותר לטפל ב-$\mathcal M$ כפרמטר מטרד.

מכיוון ש-$\mathcal M$ נכנס **ליניארית**, אפשר לבצע את האינטגרל עליו במפורש. נסמן $\mathbf r_0=\mathbf m_B-\boldsymbol\mu_{\rm th}$ ו-$\mathbf r=\mathbf r_0-\mathcal M\mathbf 1$:

$$\chi^2=\mathbf r^{\top}\mathsf C^{-1}\mathbf r=A-2\mathcal M B+\mathcal M^2 S,\quad A=\mathbf r_0^{\top}\mathsf C^{-1}\mathbf r_0,\ \ B=\mathbf 1^{\top}\mathsf C^{-1}\mathbf r_0,\ \ S=\mathbf 1^{\top}\mathsf C^{-1}\mathbf 1.$$

השלמה לריבוע נותנת $\chi^2=S(\mathcal M-B/S)^2+\left(A-B^2/S\right)$, והאינטגרל הגאוסי על $\mathcal M$ (מוקדמת אחידה) מביא

$$\boxed{\ \chi^2_{\rm marg}=A-\frac{B^2}{S}\ }$$

שאינו תלוי ב-$\mathcal M$. הקבוע $\ln(S/2\pi)$ שמתקבל תלוי רק בשגיאות המדידה ולא בפרמטרים הקוסמולוגיים, ולכן מותר להשמיט אותו. **ההיסט הטוב ביותר** הוא $\hat{\mathcal M}=B/S$, כלומר ממוצע משוקלל של השארית. מה שמרג'ינליזציה בולעת: *כל הזזה אחידה של כל הסופרנובות.* האות הקוסמולוגי הוא ש-$\mu(z)$ **משתנה** עם $z$ בצורה שונה מהנתונים.

</div>

In [ ]:
def chi2_marg(mu):
    r0 = mb - mu
    A = r0 @ (Cinv @ r0); B = s_vec @ r0
    return A - B*B/S_sum

for name, mu in models.items():
    print(f"{name:16s} χ²_marg = {chi2_marg(mu):9.1f}   M̂ = {best_offset(mu):.3f}")
t0 = time.time(); [chi2_marg(mu_model(0.3)) for _ in range(300)]; print(f"זמן לקריאה אחת של הנראות: {(time.time()-t0)/300*1e3:.2f} ms")

<div dir="rtl">

## 16. מקרה 1: $\Lambda$CDM שטוח, פרמטר אחד

מודל עם פרמטר אחד, $\Omega_m$, וכל השאר קבוע ($w=-1$). פרמטר אחד: אפשר לחשב את המאוחרת **על רשת** ולנרמל, ולהשוות ל-`emcee`. זו בדיקה מצוינת: *אם הדגימה והרשת לא מסכימות, יש באג.* (ל-`emcee` צריך `ndim≥1` ו-`nwalkers≥2·ndim`, ולכן 16 הולכים.)

</div>

In [ ]:
def log_prob_lcdm(theta):
    Om = theta[0]
    if not (0.0 < Om < 1.0): return -np.inf
    return -0.5*chi2_marg(mu_model(Om))

# --- רשת
Og = np.linspace(0.20, 0.50, 301)
lp = np.array([log_prob_lcdm([o]) for o in Og]); post = np.exp(lp - lp.max()); post /= trapezoid(post, Og)
mean_g = trapezoid(Og*post, Og); sd_g = np.sqrt(trapezoid((Og-mean_g)**2*post, Og))

# --- emcee
rng2 = np.random.default_rng(7)
p0 = 0.33 + 0.01*rng2.normal(size=(16,1))
s_l = emcee.EnsembleSampler(16, 1, log_prob_lcdm); s_l.run_mcmc(p0, 2500, progress=False)
tau_l = s_l.get_autocorr_time(tol=0)
fl = s_l.get_chain(discard=500, flat=True)[:,0]
print(f"רשת:    Ωm = {mean_g:.3f} ± {sd_g:.3f}")
print(f"emcee:  Ωm = {np.median(fl):.3f} ± {fl.std():.3f}   (τ = {tau_l[0]:.1f}, acceptance = {np.mean(s_l.acceptance_fraction):.2f})")
chi2_lcdm = chi2_marg(mu_model(Og[np.argmax(lp)])); print(f"χ²_min = {chi2_lcdm:.1f}  ל-{len(mb)-1} דרגות חופש")

fig, ax = plt.subplots(figsize=(6,3.8)); ax.hist(fl, bins=50, density=True, alpha=.5, label="emcee"); ax.plot(Og, post, "k", label="grid"); ax.set_xlabel(r"$\Omega_m$"); ax.legend(); plt.show()

<div dir="rtl">

המאוחרת כמעט גאוסית וסימטרית, והדגימה והרשת מסכימות. זה דיוק של $\sim5\%$ ב-$\Omega_m$ מסופרנובות לבד. ו-$\chi^2/\nu\approx0.88$: שימו לב שזו התאמה טובה ״מדי״ מעט, כי המטריצה כוללת את כל השגיאות השיטתיות (ולכן $\chi^2$ קטן מ-$\nu$ אינו סימן לכישלון).

</div>

<div dir="rtl">

## 17. מקרה 2: $w$CDM שטוח (שני פרמטרים)

נשחרר את משוואת המצב של האנרגיה האפלה, $w$ קבוע, עם מוקדמות אחידות $\Omega_m\in[0,1]$, $w\in[-3,0]$. עכשיו המאוחרת **אינה גאוסית**, ולכן MCMC הכרחי. נריץ עם `emcee`, נבדוק את האבחון, ונצייר corner.

</div>

In [ ]:
def log_prob_wcdm(theta):
    Om, w = theta
    if not (0.0 < Om < 1.0 and -3.0 < w < 0.0): return -np.inf
    mu = mu_model(Om, w)
    if mu is None: return -np.inf
    return -0.5*chi2_marg(mu)

best_w = minimize(lambda t: -log_prob_wcdm(t) if np.isfinite(log_prob_wcdm(t)) else 1e10, [0.3, -0.9], method="Nelder-Mead").x
print("best-fit:", best_w.round(3), " χ²_min =", round(chi2_marg(mu_model(*best_w)),2))

rng3 = np.random.default_rng(11)
p0 = best_w + np.array([0.02, 0.03])*rng3.normal(size=(32, 2))
s_w = emcee.EnsembleSampler(32, 2, log_prob_wcdm)
t0 = time.time(); s_w.run_mcmc(p0, 4000, progress=False); print(f"זמן ריצה {time.time()-t0:.0f} s")
tau_w = s_w.get_autocorr_time(tol=0); print("τ =", tau_w.round(1), "| N_steps/τ =", (4000/tau_w).round(0), "| acceptance =", np.mean(s_w.acceptance_fraction).round(2))
fw = s_w.get_chain(discard=int(5*tau_w.max()), thin=10, flat=True)
print("דגימות:", fw.shape[0], "| N_eff ≈", int((4000-5*tau_w.max())*32/tau_w.max()))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8,4.6), sharex=True)
for i, lab in enumerate([r"$\Omega_m$", r"$w$"]):
    axes[i].plot(s_w.get_chain()[:,:,i], color="k", alpha=0.2, lw=0.5); axes[i].set_ylabel(lab)
axes[1].set_xlabel("step"); plt.show()

fig = corner.corner(fw, labels=[r"$\Omega_m$", r"$w$"], levels=(0.68, 0.95), quantiles=[0.16,0.5,0.84], show_titles=True, title_fmt=".3f",
                    color="C0", plot_datapoints=False, fill_contours=True, smooth=1.0)
fig.axes[2].axhline(-1, color="C3", ls="--"); fig.axes[2].text(0.02, -0.97, r"$w=-1$", color="C3")
plt.show()

q = np.percentile(fw, [16,50,84], axis=0)
print("Ωm = %.3f +%.3f -%.3f" % (q[1,0], q[2,0]-q[1,0], q[1,0]-q[0,0]))
print("w  = %.3f +%.3f -%.3f" % (q[1,1], q[2,1]-q[1,1], q[1,1]-q[0,1]))
print("ρ(Ωm, w) = %.2f   P(w<-1) = %.2f" % (np.corrcoef(fw.T)[0,1], np.mean(fw[:,1] < -1)))

<div dir="rtl">

### קריאת התוצאה

  - **ניוון:** האליפסה צרה ומשופעת כלפי מטה ($\rho\approx-0.94$). הנתונים מודדים היטב *צירוף* של $\Omega_m$ ו-$w$: העלאת $\Omega_m$ מפצה על הורדת $w$. הכיוון הצר הוא מה שהנתונים יודעים, והארוך הוא מה שלא.
  - **צורת בננה:** הקצה מתעקל, ולכן ה-$\pm$ האסימטרי ו-$\rho$ לבדו אינם מספיקים: הפאנל עצמו מכיל יותר מידע.
  - **ניפוח שגיאה:** $\sigma_{\Omega_m}$ גדול פי 3–4 מזה של $\Lambda$CDM שטוח, כי הפרמטר הנוסף מתואם חזק איתו: $1/\sqrt{1-\rho^2}\approx2.9$.
  - **$\Lambda$CDM בתוך הפאנל:** הקו $w=-1$ עובר דרך אזור ה-$68\%$, ו-$P(w\<-1)\approx27\%$ בלבד. **אין בנתוני Pantheon+ לבדם עדות ל-$w\neq-1$.**

</div>

In [ ]:
chi2_w = chi2_marg(mu_model(*best_w))
d_chi2 = chi2_lcdm - chi2_w; N_sn = len(mb)
dBIC = (chi2_w + 2*np.log(N_sn)) - (chi2_lcdm + 1*np.log(N_sn))
print(f"Δχ² (ΛCDM → wCDM) = {d_chi2:.2f}    ΔBIC = {dBIC:+.1f}  (ln N = {np.log(N_sn):.1f})    ln B ≈ {-dBIC/2:+.1f} לטובת ΛCDM")

<div dir="rtl">

פרמטר אחד נוסף משפר את $\chi^2$ בפחות מ-$1$ (ה-״$1\sigma$״ לפרמטר נוסף), ולכן BIC מעדיף בבירור את $\Lambda$CDM ($\ln B\approx3.5$, ״חזק״ בסולם ג'פריס). **אבל** BIC הוא קירוב גס שמרומזת בו מוקדמת ספציפית: המסקנה ההגיונית היא ש-$\Lambda$CDM *מספיק*, לא שהוא *הוכח*.

</div>

<div dir="rtl">

## 18. מקרה 3: יקום עקום, ופרמטרים נגזרים

נוותר על שטיחות ונשאיר $w=-1$. הפרמטרים הנדגמים הם $(\Omega_m,\Omega_\Lambda)$, ושני גדלים **נגזרים** מחושבים שורה-שורה מהדגימות:

$$\Omega_k=1-\Omega_m-\Omega_\Lambda,\qquad q_0=\frac{\Omega_m}{2}-\Omega_\Lambda .$$

**זה היתרון הגדול של דגימה:** פרמטר נגזר אינו דורש ״התפשטות שגיאות״, פשוט מחשבים אותו לכל דגימה ומציירים היסטוגרמה, וזה תקף גם להתפלגויות לא-גאוסיות. נדרוש גם $E^2(z)\>0$ בכל $z\le2.4$ (אחרת אין פתרון פיזיקלי: היקום לא מגיע ל-$z$ הזה).

</div>

In [ ]:
def log_prob_curved(theta):
    Om, OL = theta
    if not (0.0 < Om < 1.5 and 0.0 < OL < 1.5): return -np.inf
    mu = mu_model(Om, OL=OL)
    if mu is None: return -np.inf
    return -0.5*chi2_marg(mu)

rng4 = np.random.default_rng(5)
p0 = np.array([0.30, 0.62]) + 0.02*rng4.normal(size=(32, 2))
s_c = emcee.EnsembleSampler(32, 2, log_prob_curved)
t0 = time.time(); s_c.run_mcmc(p0, 4000, progress=False); print(f"זמן ריצה {time.time()-t0:.0f} s")
tau_c = s_c.get_autocorr_time(tol=0); print("τ =", tau_c.round(1), "| acceptance =", np.mean(s_c.acceptance_fraction).round(2))
fc = s_c.get_chain(discard=int(5*tau_c.max()), thin=10, flat=True)
Om_s, OL_s = fc[:,0], fc[:,1]
derived = np.c_[Om_s, OL_s, 1-Om_s-OL_s, Om_s/2-OL_s]            # Ωm, ΩΛ, Ωk, q0
fig = corner.corner(derived, labels=[r"$\Omega_m$", r"$\Omega_\Lambda$", r"$\Omega_k$", r"$q_0$"], levels=(0.68, 0.95), quantiles=[0.16,0.5,0.84],
                    show_titles=True, title_fmt=".2f", color="C0", plot_datapoints=False, fill_contours=True, smooth=1.0)
plt.show()
for nm, v in zip(["Ωm","ΩΛ","Ωk","q0"], derived.T):
    q = np.percentile(v, [16,50,84]); print(f"{nm} = {q[1]:.3f} +{q[2]-q[1]:.3f} -{q[1]-q[0]:.3f}")
print("ρ(Ωm,ΩΛ) = %.2f   P(q0<0) = %.4f   P(Ωk>0) = %.2f" % (np.corrcoef(Om_s, OL_s)[0,1], np.mean(derived[:,3]<0), np.mean(derived[:,2]>0)))

<div dir="rtl">

**איך קוראים:**

  - **$(\Omega_m,\Omega_\Lambda)$:** סיגר משופע כלפי מעלה ($\rho\approx+0.85$). בטווח ה-$z$ הנמוך סופרנובות מודדות בעיקר את $q_0=\Omega_m/2-\Omega_\Lambda$, כלומר **צירוף אחד** של שני הפרמטרים. דרוש מידע נוסף, למשל ה-CMB שמגביל בקירוב את הסכום, וה*חיתוך* בין שני הניוונים הוא שנותן את התוצאה החדה.
  - **$q_0\<0$ באופן איתן:** תאוצת ההתפשטות היא מסקנה איתנה *במשפחת מודלים של חומר ו-$\Lambda$ עם עקמומיות כלשהי*. (זו הסתייגות חשובה: $q_0$ נגזר כאן מהמשפחה הזו ואינו מדידה ״נקייה ממודל״.)
  - **שטיחות נבחנת חלש:** $\Omega_k=0$ בתוך ה-$68\%$, כי סופרנובות אינן בוחנות עקמומיות ביעילות. ה-CMB נותן $|\Omega_k|\<0.005$.
  - **פאנלים של נגזרים:** אליפסות דקות וחד-משמעיות, כי $\Omega_k$ ו-$q_0$ הן פונקציות ליניאריות של הנדגמים. אין כאן ״מידע נוסף״, רק מבט אחר על אותו ענן.

</div>

<div dir="rtl">

## 19. בדיקה: דגימה מפורשת של $\mathcal M$

כדי לוודא שהמרג'ינליזציה האנליטית של סעיף 15 ״לא משנה דבר״, נדגום את $\mathcal M$ כפרמטר שלישי, בלי לבצע עליו את האינטגרל. אם המרג'ינליזציה נכונה, ההתפלגויות השוליות של $\Omega_m$ ו-$w$ יהיו זהות.

**שני דברים להשוות:** (1) התוצאות ($\Omega_m,w$) בשתי הדרכים; (2) $\tau$. ב-3 פרמטרים $\mathcal M$ מתואם עם $w$ ועם $\Omega_m$, ולכן הדוגם איטי יותר. **מרג'ינליזציה אנליטית, כשאפשר, משפרת גם את ביצועי הדגימה.** והעיקר: לא נכון לקבע את $\mathcal M$ ב״ערך הטוב ביותר״, כי הוא מתואם עם הפרמטרים הפיזיקליים וקיבוע היה מכווץ את השגיאות שלהם.

</div>

In [ ]:
def log_prob_wcdm_M(theta):
    Om, w, Moff = theta
    if not (0.0 < Om < 1.0 and -3.0 < w < 0.0 and -20.0 < Moff < -18.0): return -np.inf
    mu = mu_model(Om, w)
    if mu is None: return -np.inf
    r = mb - mu - Moff                                  # M כאן = היסט (מגניטודה מוחלטת + H0)
    return -0.5 * r @ (Cinv @ r)

Mhat = best_offset(mu_model(*best_w))
rng5 = np.random.default_rng(3)
p0 = np.array([best_w[0], best_w[1], Mhat]) + np.array([0.02, 0.03, 0.005])*rng5.normal(size=(32,3))
s_M = emcee.EnsembleSampler(32, 3, log_prob_wcdm_M)
t0 = time.time(); s_M.run_mcmc(p0, 8000, progress=False); print(f"זמן ריצה {time.time()-t0:.0f} s")
tau_M = s_M.get_autocorr_time(tol=0); print("τ (Ωm, w, M) =", tau_M.round(1), "   לעומת בלי M:", tau_w.round(1))
fM = s_M.get_chain(discard=int(5*tau_M.max()), thin=10, flat=True)
for lab, v1, v2 in (("Ωm", fw[:,0], fM[:,0]), ("w", fw[:,1], fM[:,1])):
    a1 = np.percentile(v1,[16,50,84]); a2 = np.percentile(v2,[16,50,84])
    print(f"{lab:3s} אנליטי: {a1[1]:.3f} +{a1[2]-a1[1]:.3f} -{a1[1]-a1[0]:.3f}    מפורש: {a2[1]:.3f} +{a2[2]-a2[1]:.3f} -{a2[1]-a2[0]:.3f}")
print(f"M^ = {np.median(fM[:,2]):.3f} ± {fM[:,2].std():.3f}   ρ(Ωm,M) = {np.corrcoef(fM[:,0], fM[:,2])[0,1]:.2f}   ρ(w,M) = {np.corrcoef(fM[:,1], fM[:,2])[0,1]:.2f}")

<div dir="rtl">

ההתפלגויות השוליות חופפות (עד רעש הדגימה), ואכן $\tau$ גדול יותר. **מרג'ינליזציה היא הטלה, ו״להתעלם״ מפרמטר מטרד פירושו לאפשר לו להשתנות.** שימו לב ש-$\mathcal M$ הוא היסט, ולא $H_0$: הערך שלו מתייחס ל-$H_0=70$ שקבענו. כדי לקבל $H_0$ צריך כיול חיצוני של $M_B$ בעזרת צפייניים, וזה בדיוק המקום שבו נולד מתח האבל.

</div>

<div dir="rtl">

## 20. סיכום, טעויות נפוצות ומה הלאה

**הזרימה המלאה, שחוזרת בכל ניתוח קוסמולוגי:**

$$\text{נתונים}\ \to\ \text{מודל}\ \to\ \ln P\ \to\ \text{הולכים}\ \to\ \texttt{run\_mcmc}\ \to\ \text{אבחון }(\tau,\hat R)\ \to\ \text{corner}\ \to\ \text{השוואת מודלים}.$$

**טבלת תקלות שכיחות:**

| תסמין                            | סיבה אפשרית                                   | מה לעשות                                            |
| -------------------------------- | --------------------------------------------- | --------------------------------------------------- |
| קבלה \< 0.1                      | פונקציה מחזירה `-inf` הרבה, או מוקדמת צרה מדי | בדקו את הערכים בנקודות התחלה; הרחיבו את המוקדמת     |
| קבלה ≈ 1                         | הולכים תקועים סביב אותה נקודה                 | פזרו את `p0` יותר                                   |
| `AutocorrError: chain too short` | `nsteps` קטן מ-$\sim50\tau$                 | הריצו יותר צעדים                                    |
| ה-corner ״נוגע בקצה״             | המוקדמת קובעת את התוצאה                       | הרחיבו אותה, או דווחו כגבול ולא כ-$\pm$            |
| השרשרת איטית מאוד                | ניוון חזק בין פרמטרים                         | שנו פרמטריזציה (מרכוז), הוסיפו מרג'ינליזציה אנליטית |
| `LinAlgError`                    | המטריצה אינה חיובית-מוגדרת                    | סימטריזציה, בדיקת `cholesky`                        |
| תוצאה שונה בין ריצות             | אין התכנסות                                   | חשבו $\hat R$, הריצו מנקודות מפוזרות               |

**מה הלאה:** שילוב עם BAO ו-CMB (שמבטלים את הניוונים), השוואת מודלים עם עדות בייסיאנית (`dynesty`, `PolyChord`), `Cobaya`/`CosmoSIS` לניתוחים עם CAMB/CLASS, ו-emulators שמחליפים נראות יקרה.

### תרגילים

1.  **ריצה מחדש:** החליפו את $w$ הקבוע ב-$w(a)=w_0+w_a(1-a)$ ודגמו $(\Omega_m,w_0,w_a)$ מהנתונים האלה. מה קורה לשגיאות?
2.  **מוקדמות:** שנו את המוקדמת של $w$ מ-$[-3,0]$ ל-$[-2,-0.5]$. האם ה-corner משתנה? ומה זה אומר על תוצאה שמוגבלת על ידי המוקדמת?
3.  **נתונים בלי שגיאות שיטתיות:** החליפו את `Csn` במטריצה אלכסונית $\mathrm{diag}(\sigma_i^2)$. איך משתנים $\chi^2_{\min}$ ו-$\sigma_{\Omega_m}$, ולמה $\chi^2$ קטן יותר *אינו* התאמה טובה יותר?
4.  **$H_0$:** כיילו את $\mathcal M$ בעזרת $M_B=-19.25$ והוציאו $H_0$ מ-$\hat{\mathcal M}$. איזה ערך מתקבל, ומה השגיאה שמתווספת מאי-הוודאות ב-$M_B$?
5.  **R̂:** חשבו $\hat R$ ל-$w$CDM מ-4 ריצות בלתי-תלויות מנקודות מפוזרות, והראו שהוא קרוב ל-1.

</div>